# Task 2 — Semantic Segmentation with Attention U-Net

**Data:** neuronal membrane dataset (ISBI 2012), 30 labelled 512x512 grayscale images.

Two facts about this dataset shape everything below:

1. **30 labelled images is tiny.** Training uses random 256x256 crops with flips and
   90-degree rotations. Evaluation uses the full 512x512 image.
2. **The official `test/` folder has no ground truth** — the labels were withheld for
   the challenge. So the metrics table cannot come from it. We split the 30 labelled
   images 20/5/5 into train/val/test, and use the official test images only for
   qualitative figures. **Say this explicitly in the report** — a teammate who reports
   metrics on the official test set has computed them against something that does not exist.

## What to write about the model (report section 1)

- **Motivation.** Plain U-Net concatenates every skip connection wholesale. The encoder's
  early layers are high-resolution but semantically poor: they carry the target *and* a
  lot of irrelevant background. The decoder has to learn to ignore that.
- **Attention gates** (Oktay et al., 2018) fix this at the skip. The coarse decoder
  feature `g` acts as a query: it and the skip `x` are each projected by a 1x1 conv,
  added, passed through ReLU, projected to a single channel and squashed with a sigmoid.
  That gives one weight per spatial position, and the skip is multiplied by it. Regions
  the decoder does not care about are suppressed *before* concatenation.
- **Difference from U-Net.** Same encoder, same decoder, same skips — only the gate is
  new. It costs ~0.35M extra parameters here (31.04M -> 31.39M, about 1%), so any gain
  is not simply "a bigger model".
- **Advantages.** No extra supervision needed, trains end to end, and the gates are
  directly visualisable (see the attention figure below) which helps explain the model.
- **Limitations.** Extra memory per skip; on a dataset where the foreground is large and
  everywhere (membranes cover ~20-25% of every image) the gates have less to suppress
  than on a task like polyp segmentation where the target is one small blob.

In [ ]:
import os
# moved into notebooks/; walk up to the repo root so src/ and data/ resolve regardless of launch cwd
while not os.path.exists("requirements.txt") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from src.attention_unet import AttentionUNet, UNet
from src.losses import LOSSES
from src.membrane import load_membrane, load_unlabelled_test
from src.results import results_table, save_run
from src.seed import get_device, set_seed
from src.segmentation import evaluate, fit

SEED = 27
set_seed(SEED)
device = get_device()
print("device:", device)

In [ ]:
CONFIG = {
    "model": "attention_unet",   # "unet" for the baseline
    "loss": "dice_bce",          # bce | dice | dice_bce | iou_bce | mse
    "crop_size": 256,
    "batch_size": 2,
    "lr": 1e-4,
    "n_epochs": 100,
    "patience": 5,
    "splits": (20, 5, 5),
    "seed": SEED,
}

train_loader, val_loader, test_loader = load_membrane(
    root="data",
    crop_size=CONFIG["crop_size"],
    batch_size=CONFIG["batch_size"],
    splits=CONFIG["splits"],
    seed=SEED,
)

In [ ]:
images, masks = next(iter(train_loader))

fig, axes = plt.subplots(2, len(images), figsize=(5 * len(images), 9), squeeze=False)
for i in range(len(images)):
    axes[0][i].imshow(images[i, 0], cmap="gray"); axes[0][i].set_title("image")
    axes[1][i].imshow(masks[i, 0], cmap="gray"); axes[1][i].set_title("mask (white = membrane)")
    for ax in (axes[0][i], axes[1][i]):
        ax.axis("off")
fig.tight_layout()

## Loss function — and why Dice + BCE

The exercise asks you to pick one and justify it. The argument for `dice_bce`:

- **BCE alone** scores every pixel independently. It gives smooth, well-behaved
  gradients from the very first step, which is what gets training off the ground. But it
  is dominated by the majority class: if ~75% of pixels are background, a model that
  predicts "background everywhere" already has a low BCE and no gradient pressure to
  find the membranes.
- **Dice alone** optimises overlap directly, so it is insensitive to that imbalance —
  it is the metric we actually report. But early in training, when predictions are near
  random, the Dice numerator is close to zero and its gradients are unstable.
- **Together** each covers the other's weak point: BCE stabilises early training, Dice
  aligns the objective with the reported metric. This is why Dice+BCE is the default in
  most medical-segmentation papers.

The `bce_weight` argument controls the mix (0.5 = equal). Note the printed foreground
percentage from the cell above — if it is close to 50%, the imbalance argument is
weaker and you should say so rather than repeat the standard line.

In [ ]:
model = (AttentionUNet() if CONFIG["model"] == "attention_unet" else UNet()).to(device)
criterion = LOSSES[CONFIG["loss"]]()
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

print(f"{CONFIG['model']}  params {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
history = fit(model, train_loader, val_loader, criterion, optimizer, device,
              n_epochs=CONFIG["n_epochs"], patience=CONFIG["patience"], scheduler=scheduler)

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(epochs, history["train_loss"], label="train")
ax1.plot(epochs, history["val_loss"], label="validation")
ax1.set(xlabel="Epoch", ylabel="Loss", title=f"{CONFIG['loss']} loss"); ax1.legend()
ax2.plot(epochs, history["train_f1"], label="train")
ax2.plot(epochs, history["val_f1"], label="validation")
ax2.set(xlabel="Epoch", ylabel="Dice / F1", title="Dice"); ax2.legend()
for ax in (ax1, ax2):
    ax.axvline(history["best_epoch"], color="grey", ls=":", lw=1)
fig.tight_layout()

In [ ]:
test_metrics, test_loss = evaluate(model, test_loader, criterion, device)
print(f"test loss {test_loss:.4f}")
for name, value in test_metrics.items():
    print(f"{name:>10}: {value:.4f}")

run_dir = save_run(tag=f"{CONFIG['model']}_{CONFIG['loss']}", config=CONFIG,
                   metrics=test_metrics, history=history, model=model)

## Qualitative results (report requirement 5)

At least two test images: original, ground truth, and prediction.

In [ ]:
model.eval()
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

with torch.no_grad():
    for col, (image, mask) in zip(range(3), test_loader):
        prob = torch.sigmoid(model(image.to(device)))[0, 0].cpu()
        axes[0][col].imshow(image[0, 0], cmap="gray"); axes[0][col].set_title("original")
        axes[1][col].imshow(mask[0, 0], cmap="gray"); axes[1][col].set_title("ground truth")
        axes[2][col].imshow(prob >= 0.5, cmap="gray"); axes[2][col].set_title("prediction")
        for row in range(3):
            axes[row][col].axis("off")
fig.tight_layout()
fig.savefig("qualitative_attention_unet.png", dpi=150)

## Attention maps

Not required by the exercise, but it is the one figure a plain-U-Net report cannot
produce, and it directly evidences your claim about what the gates do. Bright = the
gate let that region through.

In [ ]:
if CONFIG["model"] == "attention_unet":
    image, mask = next(iter(test_loader))
    with torch.no_grad():
        _, attention = model(image.to(device), return_attention=True)

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    axes[0].imshow(image[0, 0], cmap="gray"); axes[0].set_title("input")
    for ax, a in zip(axes[1:], attention):
        ax.imshow(a[0, 0].cpu(), cmap="inferno")
        ax.set_title(f"gate @ {tuple(a.shape[-2:])}")
    for ax in axes:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig("attention_maps.png", dpi=150)

## Results table (report Table 2)

In [ ]:
rows = {
    f"Attention U-Net ({CONFIG['loss']})": test_metrics,
    # "member1 model": {"jaccard": .., "f1": .., "recall": .., "precision": .., "accuracy": ..},
    # "member2 model": {...},
}
print(results_table(rows, ["jaccard", "f1", "recall", "precision", "accuracy"], "{:.4f}"))

## Loss ablation (report requirement 6: "the effect of the selected loss function")

The exercise explicitly asks you to analyse this, so run it. Set `CONFIG["loss"]` to
each of `bce`, `dice`, `dice_bce`, `iou_bce` and re-run — each call to `save_run` writes
a separate `results/` folder, so nothing is overwritten. Then compare.

What to look for: `bce` usually gives the highest **pixel accuracy** and the worst
**Jaccard**, because it is happiest predicting background. That single contrast is the
whole argument for why accuracy is a bad metric for segmentation.

## Discussion — what to actually argue

- **Attention vs plain U-Net.** Re-run with `CONFIG["model"] = "unet"`, same loss, same
  seed. The gates add ~1% parameters, so if Dice improves it is the mechanism, not capacity.
- **Accuracy is the least useful column** in Table 2. Say why: background dominates.
  Jaccard and Dice are what matter.
- **Small-data caveat.** 5 test images. Any difference of a point or two between models
  is noise. If you want to claim one model is better, run 3 seeds and report mean +- std.
- **Membranes are thin structures**, so most errors sit on boundaries. A one-pixel shift
  costs a lot of Dice while barely changing what a human would call a correct result.